# sparse_release_drawdown_control_bc_ppo_screen_v1

Notebook-first fold-local PPO/replay screen for the drawdown-control BC warm start from `sparse_release_drawdown_control_foldlocal_bc_warmstart_v1`. This notebook does **not** promote a runtime model. It asks one narrow question: after the non-leaking drawdown-control BC initialization, does a short PPO update preserve the +30pp relative / <=15% MaxDD target on held-out walk-forward tests, or does PPO drift reintroduce overfit/risk?

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_release_drawdown_control_foldlocal_bc_warmstart_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection` / BC-to-PPO handoff only; reward/model architecture are inherited from the existing short screen profile `sparse_release_bc_ppo_screen_v1`.
- **Target windows**: `wf_2022` and `wf_2023_2024`; `wf_2020` is skipped because selected labels extend into 2020 and would overlap that test year.
- **Accepted trade-off**: PPO may stop at the BC initialization if 4 iterations degrade risk. Prefer early rejection over a longer overfit branch.
- **Failure mode**: PPO improves validation while increasing held-out MaxDD, losing +30pp relative return, or failing the downtrend slice fit gate.

External basis retained for the design:

- Qlib: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO clipped policy update: https://arxiv.org/abs/1707.06347
- Learning from demonstrations / BC warm start before RL: https://arxiv.org/abs/1704.03732
- FinRL benchmark context: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Probability of Backtest Overfitting / CSCV: https://doi.org/10.21314/JCF.2016.322


In [1]:

from __future__ import annotations

import copy
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.evaluate import validation_score
from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sparse_release_drawdown_control_bc_ppo_screen_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/sparse_release_drawdown_control_foldlocal_bc_warmstart_v1'
TARGETS_PATH = SOURCE_CARD / 'selected_foldlocal_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
DOWNTREND_SLICE_ACCURACY_MIN = 0.60
DOWNTREND_SLICE_SAMPLES_MIN = 100
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'source_target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
        'target_policy': 'native_fold_local',
    },
    'wf_2023_2024': {
        'source_target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
        'target_policy': 'carry_forward_non_overlapping_train_targets',
    },
}


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

profile = load_profile(PROFILE_NAME)
source_selection = json.loads((SOURCE_CARD / 'selection.json').read_text(encoding='utf-8'))
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = targets[
        (targets['fold'] == spec['source_target_fold'])
        & (targets['datetime'] >= train_start)
        & (targets['datetime'] <= train_end)
    ].copy()
    eval_overlap = fold_targets[(fold_targets['datetime'] >= test_start) & (fold_targets['datetime'] <= test_end)]
    coverage_rows.append({
        'fold': fold_name,
        'source_target_fold': spec['source_target_fold'],
        'target_policy': spec['target_policy'],
        'target_rows': int(len(fold_targets)),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'target_start': str(fold_targets['datetime'].min().date()) if len(fold_targets) else '',
        'target_end': str(fold_targets['datetime'].max().date()) if len(fold_targets) else '',
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'eval_overlap_rows': int(len(eval_overlap)),
        'fold_local_gate': bool(len(fold_targets) > 0 and len(eval_overlap) == 0),
    })
coverage = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
    'source_decision': source_selection.get('decision'),
    'source_training_allowed': bool(source_selection.get('training_allowed', False)),
    'source_slice_fit_decision': source_selection.get('slice_fit_decision'),
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'source_bc_actor_path': str((SOURCE_CARD / 'risk_budget_usage_v1_drawdown_control_bc_actor.pt').relative_to(ROOT)),
    'windows': coverage_rows,
    'skipped_windows': [
        {
            'fold': 'wf_2020',
            'reason': 'selected drawdown-control targets include 2020 rows, so wf_2020 would overlap the held-out test year',
        }
    ],
}
pd.DataFrame(coverage_rows).to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', coverage)
coverage


{'experiment': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'source_experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
 'source_decision': 'foldlocal_bc_warmstart_ready_no_promotion',
 'source_training_allowed': True,
 'source_slice_fit_decision': 'bc_slice_fit_ready_for_ppo_screen',
 'targets_path': 'runtime/model_selection/card/sparse_release_drawdown_control_foldlocal_bc_warmstart_v1/selected_foldlocal_targets.csv',
 'feature_frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'source_bc_actor_path': 'runtime/model_selection/card/sparse_release_drawdown_control_foldlocal_bc_warmstart_v1/risk_budget_usage_v1_drawdown_control_bc_actor.pt',
 'windows': [{'fold': 'wf_2022',
   'source_target_fold': 'wf_2022_train',
   'target_policy': 'native_fold_local',
   'target_rows': 370,
   'downtrend_rows': 370,
   'target_start': '2011-06-28',
   'target_end': '2020-05-28',
   'train_start': '2010-03-11'

In [3]:

def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    out['backtest_return'] = float(bt.get('return', out.get('return', 0.0)))
    out['backtest_max_dd'] = float(bt.get('max_dd', out.get('max_dd', 0.0)))
    out['backtest_relative_return'] = float(bt.get('relative_outperformance', out.get('relative_return', 0.0)))
    return out


def prefixed_metrics(prefix: str, metrics: dict) -> dict[str, float]:
    keys = ['return', 'sharpe', 'max_dd', 'avg_pos', 'stress_avg_pos', 'buy_and_hold_return', 'relative_return']
    return {f'{prefix}_{k}': float(metrics[k]) for k in keys if k in metrics}


In [4]:

init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=200, lr=0.001, batch_size=128, seed=SEED)
weight_config = StateObjectiveBCSampleWeightConfig(
    downtrend_multiplier=2.0,
    rebound_multiplier=1.0,
    ordinary_multiplier=1.0,
    objective_weight_power=0.5,
    class_balanced_beta=0.99,
    weight_cap=20.0,
)
obs_dim = observation_dim_for_profile(profile)

fold_rows = []
history_rows = []
bc_slice_frames = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f'\n[fold:{fold_name}] target_fold={spec["source_target_fold"]} train={spec["train"]} valid={spec["valid"]} test={spec["test"]}')
    set_seed(SEED)
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = targets[
        (targets['fold'] == spec['source_target_fold'])
        & (targets['datetime'] >= train_start)
        & (targets['datetime'] <= train_end)
    ].copy()
    eval_overlap = fold_targets[(fold_targets['datetime'] >= test_start) & (fold_targets['datetime'] <= test_end)]
    if fold_targets.empty:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_fold_targets'})
        continue
    if not eval_overlap.empty:
        fold_rows.append({'fold': fold_name, 'status': 'blocked_eval_overlap', 'eval_overlap_rows': int(len(eval_overlap))})
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=weight_config,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('is_downtrend_state', 'is_rebound_cluster'),
    )
    bc_slice_summary = summarize_bc_slice_fit(
        bc_slice_metrics,
        required_slices=('is_downtrend_state',),
        min_accuracy=DOWNTREND_SLICE_ACCURACY_MIN,
        min_samples=DOWNTREND_SLICE_SAMPLES_MIN,
    )
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    test_price_series = price_frame.loc[spec['test'][0]:spec['test'][1]]
    valid_price_series = price_frame.loc[spec['valid'][0]:spec['valid'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_initial_states'})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
    bc_valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])

    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    actor.eval()
    selected_test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
    selected_valid_metrics = selection['best_metrics']
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    rel_delta = float(selected_test_metrics.get('relative_return', 0.0) - bc_test_metrics.get('relative_return', 0.0))
    dd_delta = float(selected_test_metrics.get('max_dd', 0.0) - bc_test_metrics.get('max_dd', 0.0))
    selected_gate = bool(
        selected_test_metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN
        and selected_test_metrics.get('max_dd', 1e9) <= TARGET_MAX_DD
    )
    bc_gate = bool(
        bc_test_metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN
        and bc_test_metrics.get('max_dd', 1e9) <= TARGET_MAX_DD
    )
    drift_gate = bool(rel_delta >= -5.0 and dd_delta <= 2.0)

    downtrend_slice = bc_slice_metrics.set_index('slice').loc['is_downtrend_state'].to_dict()
    fold_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'target_policy': spec['target_policy'],
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'initial_states': int(len(initial_states)),
        'target_rows': int(len(fold_targets)),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'eval_overlap_rows': int(len(eval_overlap)),
        'bc_downtrend_accuracy': float(downtrend_slice['accuracy']),
        'bc_downtrend_samples': int(downtrend_slice['samples']),
        'bc_slice_gate': bool(bc_slice_summary['slice_fit_gate']),
        'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
        'bc_after_loss': float(bc_result.metrics['after_loss']),
        'bc_weight_mean': float(bc_result.metrics.get('weight_mean', float('nan'))),
        'bc_weight_max': float(bc_result.metrics.get('weight_max', float('nan'))),
        'selected_valid_score': float(selection['best_score']),
        'selected_valid_return': float(selected_valid_metrics.get('return', 0.0)),
        'selected_valid_sharpe': float(selected_valid_metrics.get('sharpe', 0.0)),
        'selected_valid_max_dd': float(selected_valid_metrics.get('max_dd', 0.0)),
        'ppo_vs_bc_relative_delta': rel_delta,
        'ppo_vs_bc_max_dd_delta': dd_delta,
        'bc_gate': bc_gate,
        'selected_gate': selected_gate,
        'drift_gate': drift_gate,
        'fold_gate': bool(selected_gate and drift_gate and bc_slice_summary['slice_fit_gate']),
        **prefixed_metrics('bc_test', bc_test_metrics),
        **prefixed_metrics('selected_test', selected_test_metrics),
        **prefixed_metrics('bc_valid', bc_valid_metrics),
    })

fold_results = pd.DataFrame(fold_rows)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

fold_results.to_csv(CARD / 'ppo_fold_results.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

display(fold_results)
display(history)


[41333:MainThread](2026-05-26 04:57:04,603) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[41333:MainThread](2026-05-26 04:57:05,251) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[41333:MainThread](2026-05-26 04:57:05,252) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}



[fold:wf_2022] target_fold=wf_2022_train train=('2010-03-11', '2020-12-31') valid=('2021-01-01', '2021-12-31') test=('2022-01-01', '2022-12-31')


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,fold,status,target_policy,train_rows,valid_rows,test_rows,initial_states,target_rows,downtrend_rows,eval_overlap_rows,...,selected_test_stress_avg_pos,selected_test_buy_and_hold_return,selected_test_relative_return,bc_valid_return,bc_valid_sharpe,bc_valid_max_dd,bc_valid_avg_pos,bc_valid_stress_avg_pos,bc_valid_buy_and_hold_return,bc_valid_relative_return
0,wf_2022,evaluated,native_fold_local,2470,252,251,21,370,370,0,...,0.266279,-76.614257,46.438325,21.119361,1.293628,7.316687,0.253141,0.275,78.905912,-57.983405
1,wf_2023_2024,evaluated,carry_forward_non_overlapping_train_targets,2973,250,252,23,370,370,0,...,0.247183,-39.952710,41.093405,29.996339,1.657638,11.695498,0.263608,0.000,79.498558,-31.256226


,iter,score,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return,fold
0,0,44.177630,20.922507,1.217980,9.203935,0.253141,0.275,3.0,78.905912,-57.983405,wf_2022
1,4,45.517824,23.398690,1.187702,13.624207,0.309948,0.325,3.0,78.905912,-55.507222,wf_2022
2,0,90.695779,48.242332,2.202938,13.377641,0.263608,0.000,0.0,79.498558,-31.256226,wf_2023_2024
3,4,75.846464,37.107274,1.994852,9.648684,0.222522,0.000,0.0,79.498558,-42.391285,wf_2023_2024


In [5]:

evaluated = fold_results[fold_results['status'].eq('evaluated')].copy() if not fold_results.empty else pd.DataFrame()
all_fold_gate = bool((not evaluated.empty) and evaluated['fold_gate'].all() and len(evaluated) == len(WALK_FORWARD_WINDOWS))
any_selected_gate = bool((not evaluated.empty) and evaluated['selected_gate'].any())
any_bc_gate = bool((not evaluated.empty) and evaluated['bc_gate'].any())
all_slice_gate = bool((not evaluated.empty) and evaluated['bc_slice_gate'].all())
wf_2022_failed = bool((not evaluated.empty) and ('wf_2022' in set(evaluated['fold'])) and (not bool(evaluated.set_index('fold').loc['wf_2022', 'fold_gate'])))
if all_fold_gate:
    decision = 'drawdown_control_bc_ppo_screen_passed_partial_wf_no_promotion'
    next_mechanism = 'extend_to_full_fixed_split_walkforward_training_before_promotion'
elif any_selected_gate and all_slice_gate:
    decision = 'drawdown_control_bc_ppo_screen_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_risk_drift_before_longer_training'
elif any_bc_gate and all_slice_gate:
    decision = 'drawdown_control_bc_initialization_useful_ppo_drift_no_promotion'
    next_mechanism = 'retain_bc_as_checkpoint_selector_reference_or_add_ppo_risk_drift_penalty'
else:
    decision = 'drawdown_control_bc_ppo_screen_failed_no_promotion'
    next_mechanism = 'revise_foldlocal_drawdown_targets_or_state_loss_before_training'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'bc_to_ppo_handoff'],
    'promotion_allowed': False,
    'promotion_blocker': 'This is a 4-iter diagnostic PPO screen using fold-local drawdown-control targets; no actor bundle is promoted or retained.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'fold_gate_pass_count': int(evaluated['fold_gate'].sum()) if not evaluated.empty and 'fold_gate' in evaluated else 0,
    'selected_gate_pass_count': int(evaluated['selected_gate'].sum()) if not evaluated.empty and 'selected_gate' in evaluated else 0,
    'bc_gate_pass_count': int(evaluated['bc_gate'].sum()) if not evaluated.empty and 'bc_gate' in evaluated else 0,
    'bc_slice_gate_pass_count': int(evaluated['bc_slice_gate'].sum()) if not evaluated.empty and 'bc_slice_gate' in evaluated else 0,
    'wf_2022_failed': wf_2022_failed,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic states',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'ppo_fold_results.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/1704.03732',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'coverage': coverage,
    'fold_results': fold_results.to_dict(orient='records'),
    'source_summary': {
        'decision': source_summary.get('decision'),
        'selected_samples': source_summary.get('selected_readiness_row', {}).get('samples'),
        'selected_unique_actions': source_summary.get('selected_readiness_row', {}).get('unique_actions'),
        'selected_dominant_action_share': source_summary.get('selected_readiness_row', {}).get('dominant_action_share'),
        'bc_after_accuracy': source_summary.get('bc_fit_metrics', {}).get('after_accuracy'),
    },
    'conclusion': [],
}
if evaluated.empty:
    summary['conclusion'].append('No folds were evaluated; do not continue PPO from this handoff.')
else:
    for row in evaluated.to_dict(orient='records'):
        summary['conclusion'].append(
            f"{row['fold']}: BC relative={row.get('bc_test_relative_return', float('nan')):.2f}pp / MaxDD={row.get('bc_test_max_dd', float('nan')):.2f}%; "
            f"selected relative={row.get('selected_test_relative_return', float('nan')):.2f}pp / MaxDD={row.get('selected_test_max_dd', float('nan')):.2f}%; "
            f"PPO-vs-BC relative delta={row.get('ppo_vs_bc_relative_delta', float('nan')):.2f}pp, "
            f"DD delta={row.get('ppo_vs_bc_max_dd_delta', float('nan')):.2f}pp, fold_gate={bool(row.get('fold_gate', False))}."
        )
    summary['conclusion'].append(
        'No runtime promotion: this screen is shorter than full training, skips wf_2020 by leakage guard, and must be followed by full fixed-split + walk-forward validation before any actor retention.'
    )
    if wf_2022_failed:
        summary['conclusion'].append(
            'wf_2022 remains the blocker for the +30pp / <=15% MaxDD objective; use the BC path as a risk-control reference only if it beats PPO drift on MaxDD.'
        )

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
summary


{'experiment': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
 'decision': 'drawdown_control_bc_ppo_screen_failed_no_promotion',
 'selection': {'experiment': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
  'created_at': '2026-05-25T21:01:31.130097+00:00',
  'decision': 'drawdown_control_bc_ppo_screen_failed_no_promotion',
  'next_mechanism': 'revise_foldlocal_drawdown_targets_or_state_loss_before_training',
  'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
  'direct_comparison_baseline': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
  'role': 'diagnose',
  'profile': 'sparse_release_bc_ppo_screen_v1',
  'changed_layer': ['model_selection', 'bc_to_ppo_handoff'],
  'promotion_allowed': False,
  'promotion_blocker': 'This is a 4-iter diagnostic PPO screen using fold-local drawdown-control targets; no actor bundle is promoted or retained.',
  'target_relative_return_pp': 30.0,
  'target_max_dd_pct': 15.0,
  'folds_evaluated': ['wf_2022', 'wf_2023_20

In [6]:

lines = [
    f"## Runtime Conclusion: {EXPERIMENT}",
    "",
    f"- **Decision**: `{summary['decision']}`.",
    f"- **Promotion**: blocked. {selection['promotion_blocker']}",
    f"- **Source BC**: `{selection['direct_comparison_baseline']}`; source fit samples={summary['source_summary'].get('selected_samples')}, unique_actions={summary['source_summary'].get('selected_unique_actions')}, dominant_share={summary['source_summary'].get('selected_dominant_action_share'):.3f}, after_accuracy={summary['source_summary'].get('bc_after_accuracy'):.3f}.",
    f"- **Gate target**: relative return >= {TARGET_RELATIVE_RETURN:.0f}pp and MaxDD <= {TARGET_MAX_DD:.0f}% on each held-out fold.",
    "",
    "### Fold Evidence",
]
if evaluated.empty:
    lines.append("- No evaluated folds were produced.")
else:
    for row in evaluated.to_dict(orient='records'):
        lines.append(
            f"- `{row['fold']}`: BC rel={row.get('bc_test_relative_return', float('nan')):.2f}pp, "
            f"BC MaxDD={row.get('bc_test_max_dd', float('nan')):.2f}%; "
            f"selected rel={row.get('selected_test_relative_return', float('nan')):.2f}pp, "
            f"selected MaxDD={row.get('selected_test_max_dd', float('nan')):.2f}%; "
            f"PPO delta rel={row.get('ppo_vs_bc_relative_delta', float('nan')):.2f}pp, "
            f"PPO delta DD={row.get('ppo_vs_bc_max_dd_delta', float('nan')):.2f}pp, "
            f"fold_gate={bool(row.get('fold_gate', False))}."
        )
lines += [
    "",
    "### Risk Notes",
    "- This is a diagnostic 4-iteration PPO screen, not a full training branch.",
    "- `wf_2020` is intentionally skipped because the selected target table includes 2020 rows and would leak that test year.",
    "- Any validation improvement that increases held-out MaxDD is treated as PPO drift, not promotion evidence.",
    "",
    "### Artifacts",
]
for sidecar in selection['sidecars']:
    lines.append(f"- `{sidecar}`")
lines += [
    "",
    "### Next Action",
    f"- `{selection['next_mechanism']}`.",
]
display(Markdown("\n".join(lines)))


## Runtime Conclusion: sparse_release_drawdown_control_bc_ppo_screen_v1

- **Decision**: `drawdown_control_bc_ppo_screen_failed_no_promotion`.
- **Promotion**: blocked. This is a 4-iter diagnostic PPO screen using fold-local drawdown-control targets; no actor bundle is promoted or retained.
- **Source BC**: `sparse_release_drawdown_control_foldlocal_bc_warmstart_v1`; source fit samples=370, unique_actions=4, dominant_share=0.595, after_accuracy=0.900.
- **Gate target**: relative return >= 30pp and MaxDD <= 15% on each held-out fold.

### Fold Evidence
- `wf_2022`: BC rel=54.40pp, BC MaxDD=29.37%; selected rel=46.44pp, selected MaxDD=34.60%; PPO delta rel=-7.97pp, PPO delta DD=5.24pp, fold_gate=False.
- `wf_2023_2024`: BC rel=41.09pp, BC MaxDD=21.46%; selected rel=41.09pp, selected MaxDD=21.46%; PPO delta rel=0.00pp, PPO delta DD=0.00pp, fold_gate=False.

### Risk Notes
- This is a diagnostic 4-iteration PPO screen, not a full training branch.
- `wf_2020` is intentionally skipped because the selected target table includes 2020 rows and would leak that test year.
- Any validation improvement that increases held-out MaxDD is treated as PPO drift, not promotion evidence.

### Artifacts
- `coverage.csv`
- `coverage.json`
- `bc_slice_metrics.csv`
- `ppo_selection_history.csv`
- `ppo_fold_results.csv`
- `selection.json`
- `summary.json`

### Next Action
- `revise_foldlocal_drawdown_targets_or_state_loss_before_training`.